In [ ]:
# Run this once if not already installed
# !pip install pandas scikit-learn tensorflow

import re
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense

In [ ]:
df = pd.read_csv("IMDB Dataset.csv")
df = df.rename(columns={"review": "text", "sentiment": "label"})

print("Total reviews:", len(df))
print(df["label"].value_counts())
df.head()

In [ ]:
def clean_text(text):
    text = text.lower()
    text = re.sub(r"<.*?>", " ", text)       # remove HTML tags like <br/>
    text = re.sub(r"[^a-z\s]", " ", text)    # keep only letters
    text = re.sub(r"\s+", " ", text).strip() # remove extra spaces
    return text

# Test it on one example
print(clean_text("This movie was AMAZING!! <br/> 10/10 would watch again."))

In [ ]:
df["clean_text"] = df["text"].apply(clean_text)
df[["text", "clean_text"]].head(3)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    df["clean_text"], df["label"], test_size=0.2, random_state=42
)

# Convert labels to 0/1 (the model needs numbers, not words)
y_train = (y_train == "positive").astype(int)
y_test = (y_test == "positive").astype(int)

print("Train size:", len(X_train))
print("Test size:", len(X_test))

In [ ]:
VOCAB_SIZE = 10000     # only keep the top 10,000 most common words
MAX_LEN = 200           # every review will be exactly 200 numbers long

tokenizer = Tokenizer(num_words=VOCAB_SIZE, oov_token="<OOV>")
tokenizer.fit_on_texts(X_train)   # learn the vocabulary from training data only

X_train_seq = tokenizer.texts_to_sequences(X_train)
X_test_seq = tokenizer.texts_to_sequences(X_test)

X_train_pad = pad_sequences(X_train_seq, maxlen=MAX_LEN, padding="post", truncating="post")
X_test_pad = pad_sequences(X_test_seq, maxlen=MAX_LEN, padding="post", truncating="post")

print("Example review as numbers:", X_train_seq[0][:10])
print("Shape after padding:", X_train_pad.shape)

In [ ]:
model = Sequential([
    Embedding(input_dim=VOCAB_SIZE, output_dim=100),
    LSTM(64),
    Dense(1, activation="sigmoid")
])

model.compile(loss="binary_crossentropy", optimizer="adam", metrics=["accuracy"])
model.summary()

In [ ]:
history = model.fit(
    X_train_pad, y_train,
    validation_split=0.1,   # use 10% of training data to check progress
    epochs=5,
    batch_size=64
)

In [ ]:
probs = model.predict(X_test_pad)
preds = (probs > 0.5).astype(int)

acc = accuracy_score(y_test, preds)
print(f"Test Accuracy: {acc:.4f}")
print()
print(classification_report(y_test, preds, target_names=["negative", "positive"]))

In [ ]:
def predict_sentiment(sentence):
    cleaned = clean_text(sentence)
    seq = tokenizer.texts_to_sequences([cleaned])
    padded = pad_sequences(seq, maxlen=MAX_LEN, padding="post", truncating="post")
    prob = model.predict(padded)[0][0]
    label = "positive" if prob > 0.5 else "negative"
    return label, prob

# Try your own sentence here
sentence = "This movie was absolutely wonderful, I loved every minute of it!"
label, prob = predict_sentiment(sentence)
print(f"Sentence: {sentence}")
print(f"Prediction: {label} (confidence: {prob:.2f})")

In [ ]:
model.save("lstm_sentiment_model.keras")

import pickle
with open("tokenizer.pkl", "wb") as f:
    pickle.dump(tokenizer, f)

print("Model and tokenizer saved!")